# 03 — Fine-tuning eligibility check

This notebook consumes the same canonical pair file as Notebook 02 and checks whether the source supports supervised fine-tuning and meaningful held-out evaluation. The selected dataset provides explicit positive memberships only. It caps each job’s list at 30, so absence from a list is not a verified negative. To honor the no-fabricated-label requirement, this notebook does not create negatives, fit a model, or report model-quality metrics. The Django feature uses the separately bundled pretrained Sentence Transformer until an approved labeled dataset is available.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

start = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [start, *start.parents] if (p / "manage.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter inside AI_RESUME_ANALYSIS or one of its subfolders.")
sys.path.insert(0, str(PROJECT_ROOT))
pairs_path = PROJECT_ROOT / "data" / "processed" / "resume_job_positive_pairs.csv"
if not pairs_path.is_file():
    raise FileNotFoundError("Run notebooks/01_EDA_and_DATA_prep.ipynb first.")
pairs = pd.read_csv(pairs_path)
print(f"Loaded {len(pairs):,} source-labeled pairs.")
print("Observed labels:", sorted(pairs["match_label"].dropna().unique()))


In [ ]:
has_verified_negative = pairs["match_label"].isin({"explicit_irrelevant", "no_fit"}).any()
has_graded_label = "match_score" in pairs.columns and pairs["match_score"].notna().any()
if not has_verified_negative and not has_graded_label:
    training_status = "BLOCKED: source contains explicit relevant pairs only; no verified negatives or graded labels."
    print(training_status)
    print("No labels were inferred, no model was fine-tuned, and no evaluation result was generated.")
else:
    raise NotImplementedError("Review and approve the label mapping and grouped evaluation design before enabling training.")
